# baseline v3

이 베이스라인 코드는 `사전학습 모델 로드`, `배치 학습`, `파인튜닝`, `양자화`, `PEFT` 등이 적용된 버전입니다.

Colab의 GPU 환경에서 개발되었습니다.
- 런타임 - 런타임 유형 변경 - GPU로 변경(T4 GPU 등)



# 환경 준비

개발 환경에 필요한 라이브러리 버전을 고정하고 최신 버전으로 라이브러리를 업데이트합니다.

- 아래 셀 실행
- 실행 완료 후 런타임 - 세션 다시 시작

In [13]:
import torch
print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("cuDNN version:", torch.backends.cudnn.version())

Torch version: 2.10.0+cu128
CUDA version: 12.8
cuDNN version: 91002


In [14]:
!pip -q install "transformers>=4.43.2,<5.0.0" "accelerate>=0.34.2" "peft>=0.13.2" "bitsandbytes>=0.43.3" datasets pillow pandas timm --upgrade

# 데이터 준비

개발에 필요한 데이터를 준비합니다.

- train.csv, train 폴더
- test.csv, test 폴더
- sample_submission.csv

본 베이스라인은 colab에서 구글 드라이브를 마운트하여 사용합니다.

데이터를 압축 해제하는데 몇 분 정도의 시간이 소요됩니다.

#### 실습 참고 내용

    챕터 2-2 합성 데이터 실습
    - 구글 드라이브 마운트 : drive()

# 라이브러리, 데이터, 설정

In [15]:
import os, re, math, random
import pandas as pd
from PIL import Image, ImageDraw
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import torch
from typing import Dict, List, Any
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
    AutoModelForVision2Seq,
    AutoModelForZeroShotObjectDetection,
    get_scheduler,
    get_linear_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm import tqdm

# 이미지 로드 시 픽셀 제한 해제
Image.MAX_IMAGE_PIXELS = None

# 디바이스 GPU 우선 사용 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# 사전 학습 모델 정의
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
OD_MODEL_ID = "IDEA-Research/grounding-dino-base"
IMAGE_SIZE = 672
MAX_NEW_TOKENS = 8
SEED = 42
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# 데이터셋 로드
train_df = pd.read_csv("/content/train.csv")
test_df  = pd.read_csv("/content/test.csv")

# 학습데이터 200개만 추출
# train_df = train_df.sample(n=200, random_state=SEED).reset_index(drop=True)

Device: cuda


# 모델, Processor

7.5GB 정도의 모델 다운로드가 진행됩니다. 10~20분 정도가 소요됩니다.

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - LoRA 구현 : LoraConfig()

In [16]:
# # 양자화
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=False,
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16,
# )

# 프로세서
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=IMAGE_SIZE*IMAGE_SIZE,
    max_pixels=IMAGE_SIZE*IMAGE_SIZE,
    trust_remote_code=True,
)

# 사전학습 모델
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

# 양자화 모델로 로드
# base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable()

# LoRA 세팅
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM",
)

# PEFT 모델 생성
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

trainable params: 95,178,752 || all params: 8,387,345,408 || trainable%: 1.1348


In [17]:

class RecyclableObjectDetector:
    def __init__(
        self,
        model_id=OD_MODEL_ID,
        box_threshold=0.28,
        text_threshold=0.20,
        device=None
    ):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.box_threshold = box_threshold
        self.text_threshold = text_threshold

        self.processor = AutoProcessor.from_pretrained(model_id)
        self.model = AutoModelForZeroShotObjectDetection.from_pretrained(model_id).to(self.device)
        self.model.eval()

        # 세부 클래스 분류가 아니라 재활용품 후보 영역만 넓게 찾기 위한 프롬프트
        self.text_prompt = (
            "recyclable waste . recyclable item . trash item . waste object . "
            "plastic bottle . bottle . can . cup . paper box . carton . container . "
            "glass bottle . plastic container . package . bag ."
        )

    def detect(self, image_path):
        image = Image.open(image_path).convert("RGB")

        inputs = self.processor(
            images=image,
            text=self.text_prompt,
            return_tensors="pt"
        ).to(self.device)

        with torch.no_grad():
            outputs = self.model(**inputs)

        results = self.processor.post_process_grounded_object_detection(
            outputs,
            inputs.input_ids,
            box_threshold=self.box_threshold,
            text_threshold=self.text_threshold,
            target_sizes=[image.size[::-1]]
        )

        result = results[0]
        boxes = result["boxes"].detach().cpu()
        scores = result["scores"].detach().cpu()
        labels = result["labels"]

        detections = []
        for box, score, label in zip(boxes, scores, labels):
            x1, y1, x2, y2 = box.tolist()
            detections.append({
                "box": [float(x1), float(y1), float(x2), float(y2)],
                "score": float(score),
                "label": str(label)
            })
        return image, detections

    @staticmethod
    def compute_iou(box1, box2):
        x11, y11, x12, y12 = box1[:, 0:1], box1[:, 1:2], box1[:, 2:3], box1[:, 3:4]
        x21, y21, x22, y22 = box2[:, 0], box2[:, 1], box2[:, 2], box2[:, 3]

        inter_x1 = torch.maximum(x11, x21)
        inter_y1 = torch.maximum(y11, y21)
        inter_x2 = torch.minimum(x12, x22)
        inter_y2 = torch.minimum(y12, y22)

        inter_w = torch.clamp(inter_x2 - inter_x1, min=0)
        inter_h = torch.clamp(inter_y2 - inter_y1, min=0)
        inter_area = inter_w * inter_h

        area1 = (x12 - x11) * (y12 - y11)
        area2 = (x22 - x21) * (y22 - y21)
        union = area1 + area2 - inter_area

        return inter_area / (union + 1e-6)

    def merge_overlapping_boxes(self, detections, iou_threshold=0.5):
        if not detections:
            return []

        boxes = torch.tensor([d["box"] for d in detections], dtype=torch.float32)
        scores = torch.tensor([d["score"] for d in detections], dtype=torch.float32)

        keep = []
        idxs = scores.argsort(descending=True)

        while len(idxs) > 0:
            current = idxs[0].item()
            keep.append(current)

            if len(idxs) == 1:
                break

            current_box = boxes[current].unsqueeze(0)
            rest = boxes[idxs[1:]]
            ious = self.compute_iou(current_box, rest).squeeze(0)
            idxs = idxs[1:][ious < iou_threshold]

        return [detections[i] for i in keep]

    def crop_objects(self, image, detections, margin_ratio=0.08):
        w, h = image.size
        crops = []

        for det in detections:
            x1, y1, x2, y2 = det["box"]
            bw = x2 - x1
            bh = y2 - y1
            mx = bw * margin_ratio
            my = bh * margin_ratio

            nx1 = max(0, int(x1 - mx))
            ny1 = max(0, int(y1 - my))
            nx2 = min(w, int(x2 + mx))
            ny2 = min(h, int(y2 + my))

            if nx2 > nx1 and ny2 > ny1:
                crop = image.crop((nx1, ny1, nx2, ny2))
                crops.append(crop)

        return crops

    def run(self, image_path, save_dir="od_cache", iou_threshold=0.5):
        os.makedirs(save_dir, exist_ok=True)

        image, detections = self.detect(image_path)
        detections = self.merge_overlapping_boxes(detections, iou_threshold=iou_threshold)
        crops = self.crop_objects(image, detections)

        stem = os.path.splitext(os.path.basename(image_path))[0]
        crop_paths = []
        for idx, crop in enumerate(crops, start=1):
            crop_path = os.path.join(save_dir, f"{stem}_crop_{idx}.jpg")
            crop.save(crop_path)
            crop_paths.append(crop_path)

        return {
            "image_path": image_path,
            "count": len(detections),
            "detections": detections,
            "crop_paths": crop_paths
        }

In [18]:
detector = RecyclableObjectDetector(
    box_threshold=0.28,
    text_threshold=0.20,
    device=device
)
print("Object detector ready")

Object detector ready


In [19]:
def build_detected_paths(df, detector, save_dir):
    detected_paths = []
    counts = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Building {save_dir}"):
        image_path = row["path"]

        try:
            result = detector.run(image_path, save_dir=save_dir)

            # crop이 있으면 첫 crop 사용, 없으면 원본 사용
            if len(result["crop_paths"]) > 0 and os.path.exists(result["crop_paths"][0]):
                detected_path = result["crop_paths"][0]
            else:
                detected_path = image_path

            counts.append(result["count"])
        except Exception as e:
            # 절대 row drop 하지 않음
            detected_path = image_path
            counts.append(0)

        detected_paths.append(detected_path)

    out_df = df.copy()
    out_df["detected_path"] = detected_paths
    out_df["recyclable_count"] = counts

    assert len(out_df) == len(df)
    return out_df

train_df = build_detected_paths(train_df, detector, save_dir="od_cache_train")
test_df = build_detected_paths(test_df, detector, save_dir="od_cache_test")

print(train_df[["path", "detected_path", "recyclable_count"]].head())
print(test_df[["path", "detected_path", "recyclable_count"]].head())
assert len(test_df) == 5074, f"Expected 5074 test rows, got {len(test_df)}"


Building od_cache_test: 100%|██████████| 5074/5074 [08:32<00:00,  9.90it/s]

                   path         detected_path  recyclable_count
0  train/train_0001.jpg  train/train_0001.jpg                 0
1  train/train_0002.jpg  train/train_0002.jpg                 0
2  train/train_0003.jpg  train/train_0003.jpg                 0
3  train/train_0004.jpg  train/train_0004.jpg                 0
4  train/train_0005.jpg  train/train_0005.jpg                 0
                 path       detected_path  recyclable_count
0  test/test_0001.jpg  test/test_0001.jpg                 0
1  test/test_0002.jpg  test/test_0002.jpg                 0
2  test/test_0003.jpg  test/test_0003.jpg                 0
3  test/test_0004.jpg  test/test_0004.jpg                 0
4  test/test_0005.jpg  test/test_0005.jpg                 0


# 프롬프트 템플릿

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - 프롬프트 템플릿 : convert_to_chatml(), formatting_prompts_func()

In [20]:
# 모델 지시사항
# SYSTEM_INSTRUCT = (
#     "You are a helpful visual question answering assistant. "
#     "Answer using exactly one letter among a, b, c, or d. No explanation."
# )
SYSTEM_INSTRUCT = (
    "You are a professional visual analysis expert and VQA (Visual Question Answering) assistant. "
    "Analyze the provided image and question to derive the most accurate answer. "
    "You MUST output exactly one lowercase letter among 'a', 'b', 'c', or 'd'. "
    "Do not include any explanations, introductory text, or additional characters."
)

# 프롬프트
# def build_mc_prompt(question, a, b, c, d):
#     return (
#         f"{question}\n"
#         f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
#         "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요."
#     )
def build_mc_prompt(question, a, b, c, d):
    # 1. 작업 지시 및 제약 조건
    instructions = (
        "### INSTRUCTIONS:\n"
        "1. Analyze the image and choose the correct answer from the options (a) to (d).\n"
        "2. Your output must be exactly one lowercase letter: 'a', 'b', 'c', or 'd'.\n"
        "3. NEVER provide any explanation, reasoning, or filler text like 'The answer is'.\n\n"
    )

    # 2. Few-shot 예시 (모델에게 출력 형식 가이드 제공)
    examples = (
        "### EXAMPLE:\n"
        "Question: What is the color of the sky in the image?\n"
        "Options: (a) Red (b) Blue (c) Green (d) Yellow\n"
        "Answer: b\n\n"
    )

    # 3. 실제 문제 입력 및 답변 트리거
    target_task = (
        f"### ACTUAL TASK:\n"
        f"Question: {question}\n"
        f"Options:\n(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        f"Answer: "
    )

    return instructions + examples + target_task

# Custom Dataset, Collator

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - TensorDataset()

    챕터 5-2 데이터 생성 및 파인튜닝 (향후 학습 분량)
    - IntentDataset()

In [21]:
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
import numpy as np

# 1. 재활용 데이터에 특화된 증강 정의
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
])

# 커스텀 데이터셋
class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train
        self.transform = train_transform if train else None

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")

        if self.transform:
            img_np = np.array(img)
            augmented = self.transform(image=img_np)
            img = Image.fromarray(augmented['image'])

        q = str(row["question"])
        a, b, c, d = str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
        user_text = build_mc_prompt(q, a, b, c, d)

        messages = [
            {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
            {"role":"user","content":[
                {"type":"image","image":img},
                {"type":"text","text":user_text}
            ]}
        ]
        if self.train:
            gold = str(row["answer"]).strip().lower()
            messages.append({"role":"assistant","content":[{"type":"text","text":gold}]})

        return {"messages": messages, "image": img}

# 데이터 콜레이터
@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        texts, images = [], []
        for sample in batch:
            messages = sample["messages"]
            img = sample["image"]

            text = self.processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=not self.train
            )
            texts.append(text)
            images.append(img)

        enc = self.processor(
            text=texts,
            images=images,
            padding=True,
            return_tensors="pt"
        )

        if self.train:
            labels = enc["input_ids"].clone()
            for i, text in enumerate(texts):
                parts = text.split("assistant")
                if len(parts) < 2: continue

                instruction_ids = self.processor.tokenizer.encode(parts[0], add_special_tokens=False)
                mask_length = len(instruction_ids) + 1

                prompt = self.processor.apply_chat_template(
                messages[:-1],  # assistant 제외
                tokenize=False,
                add_generation_prompt=True
            )

            prompt_ids = self.processor.tokenizer.encode(
                prompt, add_special_tokens=False
            )

            labels[i, :len(prompt_ids)] = -100

            enc["labels"] = labels

        return enc

# DataLoader

#### 실습 참고 내용

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 데이터로더 정의 : DataLoader()

In [22]:
# 검증용 데이터 분리
split = int(len(train_df)*0.9)
train_subset, valid_subset = train_df.iloc[:split], train_df.iloc[split:]

# VQAMCDataset 형태로 변환
train_ds = VQAMCDataset(train_subset, processor, train=True)
valid_ds = VQAMCDataset(valid_subset, processor, train=True)

# 데이터로더
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, collate_fn=DataCollator(processor, True), num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=1, shuffle=False, collate_fn=DataCollator(processor, True), num_workers=0)

# fine-tuning

- 200개만 학습 : 10~20분 소요

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - 모델 정의 : SimpleMLP(), SequentialMLP()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [23]:
from tqdm.auto import tqdm


model = model.to(device)
GRAD_ACCUM = 16

# 옵티마이저, 학습 스케줄러
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
num_training_steps = 3 * math.ceil(len(train_loader)/GRAD_ACCUM)
scheduler = get_linear_schedule_with_warmup(optimizer, int(num_training_steps*0.03), num_training_steps)

# 스케일러
scaler = torch.cuda.amp.GradScaler(enabled=True)

# 학습 루프
global_step = 0
for epoch in range(3):
    running = 0.0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1} [train]", unit="batch")
    for step, batch in enumerate(progress_bar, start=1):
        batch = {k:v.to(device) for k,v in batch.items()}
        with torch.cuda.amp.autocast(dtype=torch.bfloat16):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        running += loss.item()

        if step % GRAD_ACCUM == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1

            avg_loss = running / GRAD_ACCUM
            progress_bar.set_postfix({"loss": f"{avg_loss:.3f}"})
            running = 0.0

    model.eval()
    val_loss = 0.0
    val_steps = 0
    with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.bfloat16):
        for vb in tqdm(valid_loader, desc=f"Epoch {epoch+1} [valid]", unit="batch"):
            vb = {k:v.to(device) for k,v in vb.items()}
            val_loss += model(**vb).loss.item()
            val_steps += 1
    print(f"[Epoch {epoch+1}] valid loss {val_loss/val_steps:.4f}")
    model.train()

# 모델 저장
SAVE_DIR = "/content/qwen2_5_vl_3b_lora"
model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
print("Saved:", SAVE_DIR)


/tmp/ipykernel_48740/1742211663.py:13: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=True)


Epoch 1 [train]:   0%|          | 0/4565 [00:00<?, ?batch/s]

/tmp/ipykernel_48740/1742211663.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16):
/tmp/ipykernel_48740/1742211663.py:43: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.bfloat16):


Epoch 1 [valid]:   0%|          | 0/508 [00:00<?, ?batch/s]

[Epoch 1] valid loss 5.2558


...

Epoch 2 [valid]:   0%|          | 0/508 [00:00<?, ?batch/s]

[Epoch 2] valid loss 5.2512


Epoch 3 [train]:   0%|          | 0/4565 [00:00<?, ?batch/s]

Epoch 3 [valid]:   0%|          | 0/508 [00:00<?, ?batch/s]

[Epoch 3] valid loss 5.2497
Saved: /content/qwen2_5_vl_3b_lora


# inference

30분~1시간 소요

#### 실습 참고 내용

    챕터4-1 RAG 기반 Customer Service AI 에이전트 개발
    - 데이터 파서 : langchain_core.output_parsers(), StrOutputParser()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [25]:
# 데이터 파서 : 모델의 응답에서 선지를 추출
def extract_choice(text: str) -> str:
    text = text.strip().lower()

    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if not lines:
        return "a"
    last = lines[-1]
    if last in ["a", "b", "c", "d"]:
        return last

    tokens = last.split()
    for tok in tokens:
        if tok in ["a", "b", "c", "d"]:
            return tok
    return "a"

# 추론을 위해 모든 레이어 활성화
model.eval()
preds = []

# 추론 루프
for i in tqdm(range(len(test_df)), desc="Inference", unit="sample"):
    row = test_df.iloc[i]
    img = Image.open(row["path"]).convert("RGB")
    user_text = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])

    messages = [
        {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
        {"role":"user","content":[
            {"type":"image","image":img},
            {"type":"text","text":user_text}
        ]}
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[img], return_tensors="pt").to(device)

    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=2, do_sample=False,
                                 eos_token_id=processor.tokenizer.eos_token_id)
    output_text = processor.batch_decode(out_ids, skip_special_tokens=True)[0]
    # print("output_text:", output_text)
    # print("extract_choice:", extract_choice(output_text))
    preds.append(extract_choice(output_text))

# 제출 파일 생성
submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
submission.to_csv("/content/submission.csv", index=False)
print("Saved /content/submission.csv")

Inference:   0%|          | 0/5074 [00:00<?, ?sample/s]

Saved /content/submission.csv


In [ ]:
# 모델 응답 예시
print(output_text)